Study peri-event Trasfer Entropy between units' spikes

In [1]:
%load_ext autoreload
%autoreload 2
import numpy as np
import fmatoolbox as fma
import regions as rg
import hoi
import ISRUtilities as isru
#import xarray as xr
import scipy.stats as sps
import statsmodels.stats.multitest as smsm
import pathlib
froot = pathlib.Path().cwd().parent.parent / 'Results/Figures/ISAUnits'
batch_file = '/mnt/hubel-data-103/Pietro/InfraSlowNRPaper/Data/IS_intervals.batch'
do_save = False

In [2]:
def rippleTimes(R):
    times = R.eventInfo('ripples')[:,2]
    return np.stack((times,fma.general.shuffleEvents(times,times[0]-1)),axis=1)

def offOnTransitions(R):
    isa = R.eventIntervals('slownr')
    on = R.eventIntervals('slowavalnr')
    _, valid = fma.general.restrict(on[:,0]-.0001,isa,s_ind=True)
    off_on = on[valid,0]
    return np.stack((off_on,fma.general.shuffleEvents(off_on,isa[0,0])),axis=1)

def _unitPETE(session,events,load_f,regs=None,when='sleep.*#0'):
    # note: no overlapping bins for time independence

    R = rg.data.Regions(session,events=events,phases=when)
    regs = R.ids if regs is None else np.asarray(regs)[np.isin(regs,R.ids)]
    spikes = R.spikes(regs=regs)
    events = load_f(R)

    n_events = 100
    peth, t, _ = fma.analysis.PETH(spikes[:,0],events[:n_events].ravel('F'),groups=spikes[:,1],limits=[-0.5,0.5],n_bins=31,fast=True) # (events, time, units)
    peth = peth.reshape(peth.shape[1],peth.shape[2],peth.shape[0]) # (time, units, events)

    model = hoi.metrics.TransferEntropy(peth[:,:,:n_events])
    fit = model.fit(method="binning", matrix=True) # (units, units, events) EXPLORE FOR DIFFERENT TAUS MAYBE
    model_sh = hoi.metrics.TransferEntropy(peth[:,:,n_events:])
    fit_sh = model_sh.fit(method="binning", matrix=True)

    units = np.full(len(fit),'none')
    for r in regs:
        units[R.units(regs=r)] = r

    return fit, fit_sh, t, units

In [3]:
# test on one session
session = fma.data.readBatchFile(batch_file)[0][10]
print(session)
te, te_sh, t, units = _unitPETE(session,'InfraSlowRhythm/infraslowaval',offOnTransitions)

/mnt/hubel-data-139/perceval/Rat003_20231226/Rat003_20231226.xml


Get list of multiplets


  0%|          |  0/1 [00:00<?,       ?it/s]

Get list of multiplets


  0%|          |  0/1 [00:00<?,       ?it/s]

In [5]:
samples = (te - te_sh).reshape(-1,te.shape[-1])
w = sps.ttest_1samp(samples,0,axis=1)

mask = np.isfinite(w.pvalue)
reject = np.full_like(w.pvalue,False,dtype=bool)
reject_valid, pvals_fdr, _, _ = smsm.multipletests(w.pvalue[mask],alpha=0.1,method='fdr_bh')
reject[mask] = reject_valid
np.sum(reject), len(reject)

(np.int64(0), 59049)

In [18]:
# slow
w = sps.wilcoxon(samples,axis=1)
w.pvalue
reject, pvals_fdr, _, _ = smsm.multipletests(w.pvalue,alpha=0.05,method='fdr_bh')

/home/pietro/uvEnvs/hoi/lib/python3.10/site-packages/scipy/stats/_wilcoxon.py:172: RuntimeWarning: invalid value encountered in divide
  z = (r_plus - mn) / se


KeyboardInterrupt: 

In [ ]:
te.mean(axis=-1) # average unit-unit TE